# GenAI Assignment 1 - full training run (Kaggle)
**Settings (right panel):** Accelerator = **GPU T4 x2** or **P100**, Internet = **On** (needs a phone-verified Kaggle account).

Then click **Save Version -> Save & Run All (Commit)**. It runs in the background (~2-3 h, `quick` profile) even if you close the browser.
When it finishes, open the version's **Output** tab and download `outputs.zip`.

In [ ]:
# 1) Get the code: either clone your GitHub repo, or attach the repo zip as a Kaggle dataset ("+ Add Input")
REPO_URL = ""   # e.g. "https://github.com/<you>/genai-assignment1.git"  (leave empty to use an attached dataset)
import os, glob, shutil, subprocess
WORK = "/kaggle/working/genai-assignment1"
if not os.path.exists(WORK):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
    else:
        hits = glob.glob("/kaggle/input/**/run_all.py", recursive=True)
        assert hits, "Attach the code as a dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(hits[0]), WORK)
os.chdir(WORK)
print(os.listdir("."))

In [ ]:
# 2) Dependencies (torch / torchvision / sklearn / pandas are preinstalled on Kaggle)
!pip install -q optuna "mlflow>=2.17,<3" onnx onnxruntime gdown
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 3) FS2K from the official Google Drive link (https://github.com/DengPingFan/FS2K)
import os, zipfile, glob
os.makedirs("/tmp/data", exist_ok=True)
if not glob.glob("/tmp/data/**/anno_train.json", recursive=True) and not glob.glob("/kaggle/input/**/anno_train.json", recursive=True):
    !gdown --id 1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp -O /tmp/data/FS2K.zip
    zipfile.ZipFile("/tmp/data/FS2K.zip").extractall("/tmp/data")
print(glob.glob("/tmp/data/**/anno_*.json", recursive=True) + glob.glob("/kaggle/input/**/anno_*.json", recursive=True))

In [ ]:
# 4) Paths and profile (Oxford-IIIT Pet downloads automatically through torchvision)
import os
os.environ["GENAI_PROFILE"] = "quick"          # smoke | quick | full
os.environ["GENAI_DATA"] = "/tmp/data"          # datasets stay out of the Output tab
os.environ["FS2K_ROOT"] = "/tmp/data"
os.environ["GENAI_OUT"] = "/kaggle/working/genai-assignment1/outputs"
os.environ["GENAI_WORKERS"] = "4" 

In [ ]:
# 5) Everything: Task 1 -> Task 2 -> Task 3 -> Task 4 -> ONNX export + verification, then zip outputs/
!python -u run_all.py --zip 2>&1 | tee train_log.txt

In [ ]:
# 6) Copy the zip + log to the top of the Output tab
import shutil
shutil.copy("/kaggle/working/genai-assignment1/outputs.zip", "/kaggle/working/outputs.zip")
shutil.copy("train_log.txt", "/kaggle/working/train_log.txt")
!ls -la /kaggle/working/genai-assignment1/outputs/onnx /kaggle/working/genai-assignment1/outputs/tables